# 🏆 Copa del Decano · Laboratorio de correlación (6 exámenes)
**Guía de 15–25 minutos, sin instalar programas.**

**Paso 1.** Desde el juego, descarga **«TODOS LOS DATOS DE LAS 5 SALAS»** (CSV).  
**Paso 2.** En Google Colab, abre **Entorno de ejecución → Ejecutar todo**. En el primer bloque elige tu CSV.  
**Paso 3.** Mira las gráficas y responde con datos: **¿el Rector debería quitar los beneficios de bienestar para aumentar la nota final?**

Este cuaderno analiza datos sintéticos creados durante el juego. Las correlaciones son **asociaciones**, no experimentos ni estimaciones de efectos causales. **Nunca se usan respuestas simuladas sin identificarlas.**


In [ ]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display, Markdown

RUTA_CSV = ""  # Opcional: nombre o ruta; déjalo vacío para seleccionar el archivo
if RUTA_CSV and Path(RUTA_CSV).is_file():
    datos = pd.read_csv(RUTA_CSV, encoding="utf-8-sig")
else:
    try:
        from google.colab import files
        print("📂 Selecciona el CSV descargado del juego (TODAS LAS SALAS).")
        subidos = files.upload()
        if not subidos: raise ValueError("No se subió ningún CSV.")
        nombre, contenido = next(iter(subidos.items()))
        datos = pd.read_csv(io.BytesIO(contenido), encoding="utf-8-sig")
    except ModuleNotFoundError:
        disponibles = sorted(Path(".").glob("*TODAS_LAS_SALAS*.csv"))
        if not disponibles:
            raise FileNotFoundError("Pon el CSV del juego junto al notebook, o escribe su ruta en RUTA_CSV.")
        datos = pd.read_csv(disponibles[-1], encoding="utf-8-sig")
datos.columns = datos.columns.str.strip()
necesarias = {"estudiante_id","sala","examen_nro","arquetipo","pga_previo","complejidad",
              "horas_estudio_jornada","horas_sueno_jornada","horas_bienestar_jornada",
              "nota_final","auto_simulado"}
faltan = sorted(necesarias - set(datos.columns))
if faltan: raise ValueError("Este no es el CSV de la Copa de 6 exámenes. Faltan: " + ", ".join(faltan))
numericas = ["sala","examen_nro","pga_previo","complejidad","horas_estudio_jornada",
             "horas_sueno_jornada","horas_bienestar_jornada","nota_final"]
for c in numericas: datos[c] = pd.to_numeric(datos[c], errors="coerce")
datos["auto_simulado"] = datos["auto_simulado"].astype(str).str.strip().str.lower().isin(["true","t","1","sí","si"])
datos = datos.dropna(subset=["estudiante_id","examen_nro","nota_final"]).copy()
display(Markdown(f"### 📦 Datos cargados: {len(datos)} registros, "
                 f"{datos.estudiante_id.nunique()} alias únicos, {datos.sala.nunique()} salas"))
display(datos.head(6))


## 1. Verifica calidad y cobertura

Cada estudiante debería tener hasta **6 exámenes**, sin repetir (sala, alias, examen). El CSV se puede descargar mientras otros siguen jugando: **no presupongas que contiene 180 filas**. Los registros **autogenerados** deben distinguirse de las decisiones tomadas manualmente.


In [ ]:
duplicados = datos.duplicated(["sala","estudiante_id","examen_nro"]).sum()
fuera_de_rango = (~datos["nota_final"].between(0,50)).sum()
horas = datos[["horas_estudio_jornada","horas_sueno_jornada","horas_bienestar_jornada"]].sum(axis=1)
exceso = (horas>14.00001).sum()
resumen_salas = (datos.groupby("sala", as_index=False)
                 .agg(estudiantes=("estudiante_id","nunique"),
                      registros=("nota_final","size"),
                      nota_media=("nota_final","mean")))
resumen_salas["examenes_esperados"] = 6*resumen_salas["estudiantes"]
resumen_salas["cobertura"] = resumen_salas["registros"]/resumen_salas["examenes_esperados"]
display(resumen_salas.round(2))
print("Repetidos:",duplicados," | Nota fuera de 0–50:",fuera_de_rango,
      " | Exceso de 14h:",exceso,
      " | Simulados:",int(datos.auto_simulado.sum()),"/",len(datos))
if duplicados or fuera_de_rango or exceso:
    print("⚠️ Revisa los datos: hay observaciones que no cumplen las reglas esperadas.")
display(datos.groupby("arquetipo", as_index=False).agg(personas=("estudiante_id","nunique"),
                                                      nota_media=("nota_final","mean")).round(2))


## 2. Correlación en tres pares de variables

**Pearson (r)** mide la intensidad y dirección de una relación *lineal*. Va de −1 a +1. **Spearman (ρ)** usa rangos y permite observar tendencias monótonas; ninguno prueba causalidad. Las gráficas muestran todos los parciales disponibles. Se excluyen valores faltantes en cada comparación.


In [ ]:
variables = {
    "Estudio": "horas_estudio_jornada",
    "Sueño": "horas_sueno_jornada",
    "Bienestar": "horas_bienestar_jornada"
}
filas=[]
fig,axs=plt.subplots(1,3,figsize=(15,4.3),sharey=True)
for ax,(nombre,col) in zip(axs,variables.items()):
    par=datos[[col,"nota_final"]].dropna()
    r=par[col].corr(par["nota_final"],method="pearson") if len(par)>=3 else np.nan
    rho=par[col].corr(par["nota_final"],method="spearman") if len(par)>=3 else np.nan
    filas.append({"factor":nombre,"n_examenes":len(par),"pearson_r":r,"spearman_rho":rho})
    ax.scatter(par[col],par["nota_final"],alpha=.48,s=23)
    if len(par)>=3 and par[col].nunique()>1:
        coef=np.polyfit(par[col],par["nota_final"],1)
        xs=np.linspace(par[col].min(),par[col].max(),70)
        ax.plot(xs,np.polyval(coef,xs),linewidth=2)
    ax.set(title=f"{nombre} → Nota | r={r:.2f}" if np.isfinite(r) else f"{nombre} → Nota",
           xlabel="Horas",ylabel="Nota / 50")
    ax.grid(alpha=.15)
plt.tight_layout();plt.show()
correlaciones=pd.DataFrame(filas)
display(correlaciones.round(3))
print("Importante: cada alumno aporta hasta seis notas; las 180 filas NO son 180 personas independientes.")


## 3. ¿Qué ocurre cuando comparamos personas con ellas mismas?

Un estudiante puede obtener notas distintas por la **complejidad del parcial** y sus decisiones de ese examen. Al restar los promedios de cada estudiante observamos una **asociación dentro de la persona**. Sigue sin ser un efecto causal, pero evita confundir parte de las diferencias *fijas* entre estudiantes con cambios dentro de ellos.


In [ ]:
centrados=datos.copy()
for x in [*variables.values(),"nota_final"]:
    centrados[x+"_intra"]=centrados[x]-centrados.groupby(["sala","estudiante_id"])[x].transform("mean")
comparaciones=[]
for nombre,col in variables.items():
    c=centrados[[col+"_intra","nota_final_intra"]].dropna()
    corr=c[col+"_intra"].corr(c["nota_final_intra"]) if len(c)>=3 else np.nan
    comparaciones.append({"factor":nombre,
        "r_entre_todos_los_examenes":correlaciones.set_index("factor").loc[nombre,"pearson_r"],
        "r_dentro_del_estudiante":corr})
display(pd.DataFrame(comparaciones).round(3))
print("Si los valores cambian bastante, el promedio de toda la Copa ocultaba diferencias individuales.")


## 4. Comprueba confusión y la simulación automática

Observa **bienestar–nota por arquetipo** y **solo respuestas manuales**. Los arquetipos tienen rangos diferentes de horas, por lo que mezclar sus decisiones puede distorsionar las correlaciones. **No interpretes que una correlación negativa significa que deban eliminarse los beneficios.**


In [ ]:
filas=[]
for grupo,d in datos.groupby("arquetipo"):
    for origen,sub in [("Todos",d),("Solo manuales",d.loc[~d.auto_simulado])]:
        par=sub[["horas_bienestar_jornada","nota_final"]].dropna()
        r=par.iloc[:,0].corr(par.iloc[:,1]) if len(par)>=3 and par.iloc[:,0].nunique()>1 else np.nan
        filas.append({"arquetipo":grupo,"origen":origen,"n":len(par),"r_bienestar_nota":r})
display(pd.DataFrame(filas).round(3))
manual=datos.loc[~datos.auto_simulado]
if len(manual)>=3:
    print("r bienestar–nota (solo manuales):",
          round(manual["horas_bienestar_jornada"].corr(manual["nota_final"]),3))
else:
    print("Sin suficientes decisiones manuales para calcular la correlación.")


## 5. Ajuste estadístico exploratorio (NO causal)

Esta regresión múltiple ajusta simultáneamente horas de estudio, sueño, bienestar, PGA previo, complejidad y arquetipo. Es una **asociación condicionada**. No identifica el efecto de quitar un beneficio: no hay asignación aleatoria de bienestar ni sabemos qué habría ocurrido a los mismos estudiantes con y sin el beneficio.


In [ ]:
xs=["horas_estudio_jornada","horas_sueno_jornada","horas_bienestar_jornada","pga_previo","complejidad"]
modelo=datos.dropna(subset=xs+["nota_final"]).copy()
Xd=pd.get_dummies(modelo[xs+["arquetipo"]],columns=["arquetipo"],drop_first=True,dtype=float)
if len(Xd)>=len(Xd.columns)+4:
    A=np.column_stack([np.ones(len(Xd)),Xd.astype(float).to_numpy()])
    b=modelo["nota_final"].to_numpy(float)
    beta=np.linalg.lstsq(A,b,rcond=None)[0]
    resultados=pd.DataFrame({"variable":["intercepto",*Xd.columns],"coeficiente_asociativo":beta})
    display(resultados.round(3))
else:
    print("Faltan observaciones para mostrar un ajuste múltiple razonable.")
print("Estos coeficientes NO son efectos de intervención (no equivalen a do(bienestar)).")


## 6. La última decisión del Rector

**Según los datos, ¿debe quitar los beneficios de bienestar para aumentar la nota final de los parciales?**  

- Describe qué muestran las **correlaciones** con las notas.
- Señala los **arquetipos y condiciones** bajo los cuales varían esas relaciones.
- Explica por qué estos datos observacionales sintéticos **no identifican** lo que ocurriría si el Rector retirara los beneficios.
- Indica qué evidencia adicional necesitaría: por ejemplo, un diseño de evaluación con **asignación aleatoria ética** o supuestos causales explícitos que puedan defenderse.

El notebook no entrega automáticamente una respuesta «sí» o «no» porque los datos no constituyen una prueba causal de esa intervención.


In [ ]:
print("DECISIÓN BASADA EN EVIDENCIA: el dataset permite explorar relaciones observadas,")
print("pero NO demuestra que QUITAR bienestar aumentaría las notas.")
print("Se requiere un diseño causal adicional antes de recomendar eliminar beneficios.")
salida="copa_decano_resumen_correlaciones.csv"
correlaciones.to_csv(salida,index=False,encoding="utf-8-sig")
print("✅ Tabla de correlaciones guardada:",salida)
try:
    from google.colab import files
    files.download(salida)
except ModuleNotFoundError:
    print("Puedes descargar el archivo desde el explorador de archivos del notebook.")
